In [ ]:
# ==========================================================
# SMART AGRICULTURE - CROP RECOMMENDATION MODEL
# Dataset Columns:
# N, P, K, temperature, humidity, ph, rainfall, label
# ==========================================================

# Install packages first (run once)
# pip install pandas numpy scikit-learn matplotlib seaborn joblib

# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ==========================================================
# LOAD DATASET
# ==========================================================

print("Loading Dataset...")

df = pd.read_csv("Crop_recommendation.csv")

print("\nFirst 5 Rows:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

# ==========================================================
# DATA CLEANING
# ==========================================================

df = df.dropna()

print("\nDataset Shape After Cleaning:")
print(df.shape)

# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

df["temp_humidity_ratio"] = (
    df["temperature"] /
    (df["humidity"] + 1)
)

df["npk_sum"] = (
    df["N"] +
    df["P"] +
    df["K"]
)

# ==========================================================
# FEATURES (X)
# ==========================================================

X = df[
    [
        "N",
        "P",
        "K",
        "temperature",
        "humidity",
        "ph",
        "rainfall",
        "temp_humidity_ratio",
        "npk_sum"
    ]
]

# ==========================================================
# TARGET (y)
# ==========================================================

y = df["label"]

# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# ==========================================================
# RANDOM FOREST MODEL
# ==========================================================

print("\nTraining Model...")

rf = RandomForestClassifier(
    random_state=42
)

# ==========================================================
# HYPERPARAMETER TUNING
# ==========================================================

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None]
}

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# ==========================================================
# BEST MODEL
# ==========================================================

model = grid_search.best_estimator_

print("\nBest Parameters:")
print(grid_search.best_params_)

# ==========================================================
# PREDICTIONS
# ==========================================================

predictions = model.predict(X_test)

# ==========================================================
# ACCURACY
# ==========================================================

accuracy = accuracy_score(
    y_test,
    predictions
)

print("\nModel Accuracy:")
print(f"{accuracy * 100:.2f}%")

# ==========================================================
# CLASSIFICATION REPORT
# ==========================================================

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions
    )
)

# ==========================================================
# CONFUSION MATRIX
# ==========================================================

cm = confusion_matrix(
    y_test,
    predictions
)

print("\nConfusion Matrix:")
print(cm)

# ==========================================================
# VISUALIZATION
# ==========================================================

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=False,
    cmap="Blues"
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

# ==========================================================
# SAVE MODEL
# ==========================================================

joblib.dump(
    model,
    "crop_recommendation_model.pkl"
)

print("\nModel Saved Successfully!")
print("File: crop_recommendation_model.pkl")

# ==========================================================
# SAMPLE PREDICTION
# ==========================================================

print("\nTesting Sample Prediction...")

sample_N = 90
sample_P = 42
sample_K = 43
sample_temperature = 20.8
sample_humidity = 82.0
sample_ph = 6.5
sample_rainfall = 202.9

sample_temp_humidity_ratio = (
    sample_temperature /
    (sample_humidity + 1)
)

sample_npk_sum = (
    sample_N +
    sample_P +
    sample_K
)

sample_data = [[
    sample_N,
    sample_P,
    sample_K,
    sample_temperature,
    sample_humidity,
    sample_ph,
    sample_rainfall,
    sample_temp_humidity_ratio,
    sample_npk_sum
]]

predicted_crop = model.predict(
    sample_data
)

print("\nRecommended Crop:")
print(predicted_crop[0])

# ==========================================================
# LOAD MODEL LATER
# ==========================================================

loaded_model = joblib.load(
    "crop_recommendation_model.pkl"
)

print("\nSaved model loaded successfully!")

# ==========================================================
# PROJECT COMPLETE
# ==========================================================

print("\n======================================")
print("PROJECT COMPLETED SUCCESSFULLY")
print("======================================")
print("Output Files Generated:")
print("1. crop_recommendation_model.pkl")
print("======================================")